# Distributed ML pipeline — PySpark / Spark MLlib (Databricks Free Edition)

Replication of the household **flood** and **landslide** risk classifiers from
*Quiliche, Maçaira, Araujo Baião & Leiras (2025), IJDRR 127, 105593* —
rebuilt with **distributed Spark MLlib** instead of single-node `XGBoost` + `BayesSearchCV`.

This notebook runs **both hazards** in one pass and writes a metrics table that the companion
notebook `automl_flaml_comparison.ipynb` reads to compare against AutoML.

**What changed vs. the paper's pandas/sklearn code, and why**

| pandas / sklearn | distributed Spark | reason |
|---|---|---|
| `pd.read_csv` | `spark.read.csv` | data stays a distributed DataFrame |
| `RobustScaler` (sklearn) | `pyspark.ml.feature.RobustScaler` | native, `withCentering=True` matches sklearn |
| feature matrix `X` | `VectorAssembler` → one `features` vector | MLlib models take a single vector column |
| `scale_pos_weight` | per-row `weightCol` | MLlib's way to do cost-sensitive learning |
| `XGBClassifier` | `GBTClassifier` | MLlib's distributed gradient boosting |
| `BayesSearchCV` (skopt), stratified 10-fold, maximizing GIBA | explicit grid loop on one 80/20 split, **maximizing the same GIBA** | PySpark has no Bayesian search, its built-in evaluators cannot score GIBA, and k-fold overflows the serverless model cache |
| GIBA / GMean / specificity | confusion matrix via Spark aggregation | custom imbalance metrics, computed distributed |
| SHAP | `.featureImportances` + Spark PDP | SHAP is single-node only |
| `pickle.dump` | `model.write().save(...)` | MLlib-native model format |

**Serverless (Spark Connect) gotchas handled below:** a scratch Volume path for MLlib caching,
Delta-safe column names (no spaces), and a lightweight search to stay under the 1 GB model-cache limit.

## 0. Data setup *(run once)*

Creates the Unity Catalog Volume and copies the 8 CSVs from a Workspace folder into it.
Edit `SRC` to the Workspace folder where you uploaded the CSVs. Skip this cell if the files are
already in the Volume.

In [0]:
import glob, os, shutil

SRC = "/Workspace/Users/renato.quiliche@pucp.pe/thesis_data"
DST = "/Volumes/workspace/default/thesis_data/Databases/TrainTestSplit"

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.thesis_data")
os.makedirs(DST, exist_ok=True)

files = glob.glob(f"{SRC}/**/*.csv", recursive=True)
for f in files:
    shutil.copy(f, DST)

print(len(files), "files copied:", sorted(os.listdir(DST)))

8 files copied: ['xf_test.csv', 'xf_train.csv', 'xl_test.csv', 'xl_train.csv', 'yf_test.csv', 'yf_train.csv', 'yl_test.csv', 'yl_train.csv']


## 1. Configuration

Set your real Unity Catalog names. On Free Edition the defaults are `workspace` / `default`.

In [0]:
# --- edit these to match your workspace ---
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "thesis_data"

LABELS = ["flood_risk", "landslide_risk"]
PREFIX = {"flood_risk": ("xf", "yf"), "landslide_risk": ("xl", "yl")}

# --- derived paths (no need to edit) ---
BASE    = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATA    = f"{BASE}/Databases/TrainTestSplit"     # where the 8 split CSVs live
TMP     = f"{BASE}/sparkml_tmp"                  # MLlib scratch space
MODELS  = f"{BASE}/models"                       # saved models
RESULTS = f"{BASE}/results"                      # metrics tables shared with the AutoML notebook
print("reading from", DATA)

reading from /Volumes/workspace/default/thesis_data/Databases/TrainTestSplit


## 2. SparkML scratch path  *(required on serverless)*

MLlib's tuning caches intermediate DataFrames as Delta. On serverless there is no local disk it may
use, so it requires a **UC Volume** path via the `SPARKML_TEMP_DFS_PATH` environment variable.
This must run **before any `.fit()`**. If you restart the notebook, run this cell again.

In [0]:
import os
dbutils.fs.mkdirs(TMP)
dbutils.fs.mkdirs(RESULTS)
os.environ["SPARKML_TEMP_DFS_PATH"] = TMP
print("SparkML temp path:", os.environ["SPARKML_TEMP_DFS_PATH"])

SparkML temp path: /Volumes/workspace/default/thesis_data/sparkml_tmp


## 3. Load data, attach labels, clean names, add weights

The label CSVs (`yf_*.csv`, `yl_*.csv`) are a single column with **no join key** — they align to the
feature rows **by position only**. To reunite them in Spark without pandas: read each in a
**single partition** (`coalesce(1)`) so `monotonically_increasing_id()` is `0..n-1` in file order,
then join on that id. Safe here because both files were written together in identical order and the
data is small.

The region dummy columns (from pandas `get_dummies`) contain **spaces** (`13La Libertad`, …), which
Delta rejects, so all names are sanitized. A **`weight`** column (`n_neg / n_pos` for positives) gives
cost-sensitive learning — the `scale_pos_weight` analog.

In [0]:
import re
from pyspark.sql import functions as F

ID_COLS = ["conglome", "vivienda", "hogar", "year"]

def read(name):
    return (spark.read.option("header", True).option("inferSchema", True)
                 .csv(f"{DATA}/{name}.csv")
                 .coalesce(1)
                 .withColumn("_rid", F.monotonically_increasing_id()))

def clean_cols(df):                      # Delta-safe column names
    for c in df.columns:
        c2 = re.sub(r'[ ,;{}()\n\t=]', '_', c)
        if c2 != c:
            df = df.withColumnRenamed(c, c2)
    return df

def load(label):
    xp, yp = PREFIX[label]
    train = clean_cols(read(f"{xp}_train").join(read(f"{yp}_train"), "_rid").drop("_rid"))
    test  = clean_cols(read(f"{xp}_test").join(read(f"{yp}_test"),  "_rid").drop("_rid"))

    feature_cols = [c for c in train.columns if c not in ID_COLS + [label]]
    assert len(feature_cols) == 112, f"expected 112 features, got {len(feature_cols)}"

    counts = {r[label]: r["count"] for r in train.groupBy(label).count().collect()}
    pos_weight = counts[0] / counts[1]   # scale_pos_weight = n_neg / n_pos
    add_weight = lambda df: df.withColumn(
        "weight", F.when(F.col(label) == 1, F.lit(pos_weight)).otherwise(F.lit(1.0)))
    return add_weight(train), add_weight(test), feature_cols, pos_weight, counts

## 4. The paper's imbalance metrics

Computed from a **distributed confusion matrix** (one Spark aggregation), then the scalar math on the
driver. Includes **GIBA** (the paper's tuning objective), G-Mean, sensitivity, specificity, MCC,
balanced accuracy and AUC.
Predictions use the default 0.5 probability threshold.

In [0]:
import math

def giba(tpr, tnr):
    # The paper's tuning objective, as its code computes it:
    #   imblearn make_index_balanced_accuracy(alpha=0.9, squared=True)(geometric_mean_score)
    # With that call the dominance term is zero (sensitivity and specificity are macro-averaged),
    # so the score equals G-Mean squared = sensitivity x specificity.
    return tpr * tnr

def metrics_from_counts(TP, TN, FP, FN, auc):
    tpr  = TP/(TP+FN) if TP+FN else 0.0     # sensitivity / recall
    tnr  = TN/(TN+FP) if TN+FP else 0.0     # specificity
    prec = TP/(TP+FP) if TP+FP else 0.0
    gms  = math.sqrt(tpr*tnr)
    den  = math.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    mcc  = (TP*TN - FP*FN)/den if den else 0.0
    f1   = 2*prec*tpr/(prec+tpr) if prec+tpr else 0.0
    return {"AUC": auc, "Sensitivity": tpr, "Specificity": tnr, "BalancedAcc": (tpr+tnr)/2,
            "GMean": gms, "GIBA": giba(tpr, tnr), "MCC": mcc, "Precision": prec, "F1": f1,
            "TP": TP, "TN": TN, "FP": FP, "FN": FN}

from pyspark.ml.evaluation import BinaryClassificationEvaluator

def imbalance_metrics(model, df, label):
    pred = model.transform(df)
    r = pred.select(
        F.sum(((F.col(label)==1)&(F.col("prediction")==1)).cast("int")).alias("TP"),
        F.sum(((F.col(label)==0)&(F.col("prediction")==0)).cast("int")).alias("TN"),
        F.sum(((F.col(label)==0)&(F.col("prediction")==1)).cast("int")).alias("FP"),
        F.sum(((F.col(label)==1)&(F.col("prediction")==0)).cast("int")).alias("FN"),
    ).first()
    auc = BinaryClassificationEvaluator(labelCol=label, metricName="areaUnderROC").evaluate(pred)
    return metrics_from_counts(r["TP"], r["TN"], r["FP"], r["FN"], auc)

## 5. Pipeline and hyperparameter search — grid search on the paper's objective

**What kind of search is this?** A **grid search** (4 combinations) scored on **one random 80/20
train/validation split**. Each candidate is scored by **GIBA at the 0.5 threshold — the same objective
the paper maximized** — and the winner is refit on all training data. The paper used Bayesian search
(50 iterations) with stratified 10-fold CV on that objective.

**Why an explicit loop instead of `TrainValidationSplit`?** MLlib's built-in evaluators only offer
AUC-type metrics for binary classification, so they cannot score GIBA. The loop below fits each
candidate, scores it with the same distributed confusion matrix used for the final evaluation, and
releases the model before fitting the next one, which also keeps the serverless model cache small.

> If you hit `ML_CACHE_SIZE_OVERFLOW`, run `dbutils.library.restartPython()` to clear models from
> earlier runs, then re-run from section 1.

In [0]:
import time
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, RobustScaler
from pyspark.ml.classification import GBTClassifier

GRID = [{"maxDepth": d, "maxIter": n} for d in (4, 6) for n in (40, 60)]   # 4 combos

def make_pipeline(label, feature_cols, params):
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
    scaler    = RobustScaler(inputCol="features_raw", outputCol="features",
                             withCentering=True, withScaling=True)
    gbt       = GBTClassifier(labelCol=label, featuresCol="features", weightCol="weight",
                              seed=0, **params)
    return Pipeline(stages=[assembler, scaler, gbt])

def tune(train, label, feature_cols):
    fit_part, val_part = train.randomSplit([0.8, 0.2], seed=0)   # single 80/20 split

    t0 = time.time()
    scores = []
    for params in GRID:
        model = make_pipeline(label, feature_cols, params).fit(fit_part)
        m = imbalance_metrics(model, val_part, label)
        scores.append(m["GIBA"])
        print(f"  {params} -> validation GIBA {m['GIBA']:.4f} "
              f"(sensitivity {m['Sensitivity']:.3f}, specificity {m['Specificity']:.3f})")
        del model                                  # release it from the serverless model cache

    best_params = GRID[max(range(len(GRID)), key=scores.__getitem__)]
    best = make_pipeline(label, feature_cols, best_params).fit(train)   # refit on all training data
    return best, best_params, max(scores), time.time() - t0

## 6. One experiment = load → tune → evaluate → interpret → save

`featureImportances` replaces SHAP's mean-|value| ranking. For a **partial-dependence-style** view that
stays in Spark, the altitude feature is binned and the predicted positive-class probability is
averaged per bin.

In [0]:
from pyspark.ml.functions import vector_to_array

def run_experiment(label):
    train, test, feature_cols, pos_weight, counts = load(label)
    print(f"===== {label} =====")
    print("train rows:", counts[0] + counts[1], "| positives:", counts[1],
          "| pos_weight:", round(pos_weight, 2), "| test rows:", test.count())

    best, params, val_giba, seconds = tune(train, label, feature_cols)
    print("best params:", params, "| validation GIBA:", round(val_giba, 4),
          "| search time:", round(seconds, 1), "s")

    m = imbalance_metrics(best, test, label)
    print(f"--- Test-set metrics ({label}, 2021 hold-out) ---")
    for k, v in m.items():
        print(f"{k:>12}: {round(v, 4) if isinstance(v, float) else v}")

    imp = best.stages[-1].featureImportances
    rows = sorted(((feature_cols[i], float(imp[i])) for i in range(len(feature_cols))),
                  key=lambda t: -t[1])[:15]
    print("Top 15 features:")
    spark.createDataFrame(rows, ["feature", "importance"]).show(truncate=False)

    pred = best.transform(test).withColumn("p", vector_to_array("probability")[1])
    print("Mean predicted risk by altitude band:")
    (pred.withColumn("altitude_bin", (F.col("altitude")/500).cast("int")*500)
         .groupBy("altitude_bin").agg(F.avg("p").alias("avg_risk"), F.count("*").alias("n"))
         .orderBy("altitude_bin").show())

    model_path = f"{MODELS}/gbt_{label}"
    best.write().overwrite().save(model_path)
    print("saved to:", model_path)

    return {"label": label, "model": "Spark MLlib GBT", "search": "grid, 4 combos, 80/20 split, objective GIBA",
            "best_config": str(params), "search_seconds": round(seconds, 1),
            "validation_giba": val_giba, **m}

results = {}

## 7. Floods

In [0]:
results["flood_risk"] = run_experiment("flood_risk")

===== flood_risk =====
train rows: 72952 | positives: 1020 | pos_weight: 70.52 | test rows: 28163
  {'maxDepth': 4, 'maxIter': 40} -> validation GIBA 0.5740 (sensitivity 0.703, specificity 0.817)
  {'maxDepth': 4, 'maxIter': 60} -> validation GIBA 0.5849 (sensitivity 0.708, specificity 0.826)
  {'maxDepth': 6, 'maxIter': 40} -> validation GIBA 0.5536 (sensitivity 0.638, specificity 0.868)
  {'maxDepth': 6, 'maxIter': 60} -> validation GIBA 0.5537 (sensitivity 0.627, specificity 0.883)
best params: {'maxDepth': 4, 'maxIter': 60} | validation GIBA: 0.5849 | search time: 269.2 s
--- Test-set metrics (flood_risk, 2021 hold-out) ---
         AUC: 0.8145
 Sensitivity: 0.6717
 Specificity: 0.8312
 BalancedAcc: 0.7515
       GMean: 0.7472
        GIBA: 0.5583
         MCC: 0.1113
   Precision: 0.0274
          F1: 0.0527
          TP: 133
          TN: 23245
          FP: 4720
          FN: 65
Top 15 features:
+----------------------+--------------------+
|feature               |importance    

## 8. Landslides

In [0]:
results["landslide_risk"] = run_experiment("landslide_risk")

===== landslide_risk =====
train rows: 72952 | positives: 3931 | pos_weight: 17.56 | test rows: 28163
  {'maxDepth': 4, 'maxIter': 40} -> validation GIBA 0.6164 (sensitivity 0.867, specificity 0.711)
  {'maxDepth': 4, 'maxIter': 60} -> validation GIBA 0.6211 (sensitivity 0.867, specificity 0.717)
  {'maxDepth': 6, 'maxIter': 40} -> validation GIBA 0.6125 (sensitivity 0.836, specificity 0.733)
  {'maxDepth': 6, 'maxIter': 60} -> validation GIBA 0.6177 (sensitivity 0.831, specificity 0.743)
best params: {'maxDepth': 4, 'maxIter': 60} | validation GIBA: 0.6211 | search time: 252.3 s
--- Test-set metrics (landslide_risk, 2021 hold-out) ---
         AUC: 0.8734
 Sensitivity: 0.871
 Specificity: 0.7332
 BalancedAcc: 0.8021
       GMean: 0.7991
        GIBA: 0.6386
         MCC: 0.3032
   Precision: 0.1626
          F1: 0.274
          TP: 1377
          TN: 19489
          FP: 7093
          FN: 204
Top 15 features:
+----------------------+--------------------+
|feature               |import

## 9. Summary table

Both hazards side by side, saved to the Volume so the AutoML notebook can load it as the baseline.

In [0]:
import pandas as pd

summary = pd.DataFrame([results[l] for l in LABELS if l in results])
summary.to_csv(f"{RESULTS}/spark_gbt_metrics.csv", index=False)
print("saved:", f"{RESULTS}/spark_gbt_metrics.csv")

cols = ["label", "AUC", "Sensitivity", "Specificity", "BalancedAcc", "GMean", "GIBA", "MCC",
        "Precision", "F1", "TP", "FN", "FP", "TN", "best_config", "search_seconds"]
print(summary[cols].round(4).to_string(index=False))

saved: /Volumes/workspace/default/thesis_data/results/spark_gbt_metrics.csv
         label    AUC  Sensitivity  Specificity  BalancedAcc  GMean   GIBA    MCC  Precision     F1   TP  FN   FP    TN                    best_config  search_seconds
    flood_risk 0.8145       0.6717       0.8312       0.7515 0.7472 0.5583 0.1113     0.0274 0.0527  133  65 4720 23245 {'maxDepth': 4, 'maxIter': 60}           269.2
landslide_risk 0.8734       0.8710       0.7332       0.8021 0.7991 0.6386 0.3032     0.1626 0.2740 1377 204 7093 19489 {'maxDepth': 4, 'maxIter': 60}           252.3


## Appendix — alternatives

**Want k-fold CV instead?** Wrap the loop in `tune` over several folds — but only on a real cluster;
it trains `numFolds ×` more models, which is slow on Free Edition's small serverless compute.

**Want a true Bayesian search?** PySpark has none built in. Use **Optuna** on the driver, looping over
`pipeline.fit()` and reading the validation GIBA each trial. (Skip `hyperopt.SparkTrials` — it needs a classic cluster.)

**Want actual XGBoost on Spark?** `from xgboost.spark import SparkXGBClassifier` — a drop-in for the last
pipeline stage (`features_col`, `label_col`, `weight_col`, `num_workers=...`). It targets **classic
clusters**; on Free Edition serverless it usually fails because Spark Connect hides the RDD / barrier-mode
APIs it relies on. If so, stay on `GBTClassifier`.

**Want AutoML?** See `automl_flaml_comparison.ipynb`.